In [ ]:
pip install tensorflow opencv-python scikit-learn matplotlib kagglehub


In [ ]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
import os
import numpy as np
import cv2
import tensorflow as tf
import kagglehub
import matplotlib.pyplot as plt

from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras.layers import GlobalMaxPooling2D, Concatenate, Dropout


In [ ]:
BATCH_SIZE = 32
EPOCHS = 15

In [ ]:
dataset_path = kagglehub.dataset_download("kshitizbhargava/deepfake-face-images")

print("Dataset path:", dataset_path)
print("Level 1 folders:", os.listdir(dataset_path))

DATASET_PATH = os.path.join(dataset_path, "Final Dataset")

print("Using DATASET_PATH:", DATASET_PATH)
print("Level 2 folders:", os.listdir(DATASET_PATH))

Using Colab cache for faster access to the 'deepfake-face-images' dataset.
Dataset path: /kaggle/input/deepfake-face-images
Level 1 folders: ['Final Dataset']
Using DATASET_PATH: /kaggle/input/deepfake-face-images/Final Dataset
Level 2 folders: ['Fake', 'Real', 'dataset.csv']


In [ ]:
full_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary',
    validation_split=0.2,
    subset="training",
    seed=42
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary',
    validation_split=0.2,
    subset="validation",
    seed=42
)

Found 12890 files belonging to 2 classes.
Using 10312 files for training.
Found 12890 files belonging to 2 classes.
Using 2578 files for validation.


In [ ]:
print(os.listdir(DATASET_PATH))

['Fake', 'Real', 'dataset.csv']


In [ ]:
val_batches = tf.data.experimental.cardinality(val_ds)
test_ds = val_ds.take(val_batches // 2)
val_ds = val_ds.skip(val_batches // 2)

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

full_ds = full_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE

full_ds = full_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
base_model = Xception(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = True

x = GlobalAveragePooling2D()(base_model.output)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(1, activation='sigmoid', dtype='float32')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    full_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop]
)

Epoch 1/15
323/323 ━━━━━━━━━━━━━━━━━━━━ 493s 905ms/step - accuracy: 0.8575 - loss: 0.3258 - val_accuracy: 0.8998 - val_loss: 0.3979
Epoch 2/15
323/323 ━━━━━━━━━━━━━━━━━━━━ 77s 237ms/step - accuracy: 0.9633 - loss: 0.1034 - val_accuracy: 0.8991 - val_loss: 0.6541
Epoch 3/15
323/323 ━━━━━━━━━━━━━━━━━━━━ 76s 235ms/step - accuracy: 0.9700 - loss: 0.0867 - val_accuracy: 0.9022 - val_loss: 0.4712
Epoch 4/15
323/323 ━━━━━━━━━━━━━━━━━━━━ 82s 235ms/step - accuracy: 0.9834 - loss: 0.0477 - val_accuracy: 0.9545 - val_loss: 0.1298
Epoch 5/15
323/323 ━━━━━━━━━━━━━━━━━━━━ 76s 234ms/step - accuracy: 0.9872 - loss: 0.0348 - val_accuracy: 0.9877 - val_loss: 0.0453
Epoch 6/15
323/323 ━━━━━━━━━━━━━━━━━━━━ 75s 232ms/step - accuracy: 0.9937 - loss: 0.0176 - val_accuracy: 0.9106 - val_loss: 0.1741
Epoch 7/15
323/323 ━━━━━━━━━━━━━━━━━━━━ 75s 233ms/step - accuracy: 0.9916 - loss: 0.0302 - val_accuracy: 0.9823 - val_loss: 0.0628
Epoch 8/15
323/323 ━━━━━━━━━━━━━━━━━━━━ 75s 233ms/step - accuracy: 0.9897 - loss: 

In [ ]:
def evaluate_dataset(dataset, name):
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.flatten())

    y_true = np.array(y_true)
    y_pred_binary = (np.array(y_pred) > 0.5).astype(int)

    print(f"\n=== {name} PERFORMANCE ===")
    print("Accuracy:", accuracy_score(y_true, y_pred_binary))
    print("Precision:", precision_score(y_true, y_pred_binary))
    print("Recall:", recall_score(y_true, y_pred_binary))
    print("F1-score:", f1_score(y_true, y_pred_binary))
    print("AUC:", roc_auc_score(y_true, y_pred))


evaluate_dataset(test_ds, "CLEAN TEST")

def degrade_resolution(img, scale=0.5):
    h, w = img.shape[:2]
    img_small = cv2.resize(img, (int(w*scale), int(h*scale)))
    return cv2.resize(img_small, (w, h))

def add_gaussian_noise(img, sigma=20):
    noise = np.random.normal(0, sigma, img.shape)
    noisy = img + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def apply_blur(image, severity):
    k = int(3 + severity * 10)
    if k % 2 == 0:
        k += 1
    image = cv2.GaussianBlur(image, (k, k), 0)
    return image

def evaluate_distortion(distortion_function, name):
    y_true = []
    y_pred = []

    for images, labels in test_ds:
        images_np = images.numpy() * 255
        distorted_images = []

        for img in images_np:
            distorted = distortion_function(img.astype(np.uint8))
            distorted = distorted / 255.0
            distorted_images.append(distorted)

        distorted_images = np.array(distorted_images)
        preds = model.predict(distorted_images)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.flatten())

    y_true = np.array(y_true)
    y_pred_binary = (np.array(y_pred) > 0.5).astype(int)

    print(f"\n=== {name} PERFORMANCE ===")
    print("Accuracy:", accuracy_score(y_true, y_pred_binary))
    print("Precision:", precision_score(y_true, y_pred_binary))
    print("Recall:", recall_score(y_true, y_pred_binary))
    print("F1-score:", f1_score(y_true, y_pred_binary))
    print("AUC:", roc_auc_score(y_true, y_pred))


evaluate_distortion(lambda img: degrade_resolution(img, 0.5), "RESOLUTION DEGRADATION")
evaluate_distortion(lambda img: add_gaussian_noise(img, 0.01), "GAUSSIAN NOISE")
evaluate_distortion(lambda img: apply_blur(img, 0.2), "BLUR")


1/1 ━━━━━━━━━━━━━━━━━━━━ 20s 20s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0

In [ ]:
model.save("xception_robustness_model.h5")

print("\nTraining and robustness evaluation complete.")


Training and robustness evaluation complete.
